In [20]:
import gc
import json
import os, h5py
import math
import multiprocessing
import numpy as np
import pandas as pd
import torch
import importlib
import logging
from pathlib import Path
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.utils import resample
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
mp.set_start_method('spawn', force=True)

# Pycox and PyTorch tuples for survival analysis
import torchtuples as tt
import pycox
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from pycox.models import CoxPH, DeepHit
from pycox.evaluation import EvalSurv

# Ray for hyperparameter tuning and distributed processing
import ray
from ray import tune
from ray.tune import CLIReporter
from ray.tune.search.bayesopt import BayesOptSearch
from ray.tune.search.optuna import OptunaSearch
from ray.tune.search import ConcurrencyLimiter
from ray.tune.schedulers import ASHAScheduler, PopulationBasedTraining
from ray.air import session
import ray.cloudpickle as pickle

# Custom modules for data handling, balancing, training, evaluation, and model architectures
import dataloader2
import databalancer2
import datatrainer2
import modeleval
import netweaver2

# Reload custom modules to ensure latest changes are available
importlib.reload(dataloader2)
importlib.reload(databalancer2)
importlib.reload(datatrainer2)
importlib.reload(modeleval)
importlib.reload(netweaver2)

# Import specific functions from custom modules to keep code clean and readable
from netweaver2 import (
    lstm_net_init, DHANNWrapper, LSTMWrapper, generalized_ann_net_init
)
from dataloader2 import (
    load_and_transform_data, preprocess_data #stack_sequences, dh_dataset_loader
)
from databalancer2 import (
    define_medoid_general, df_event_focus, rebalance_data, underbalance_data_general, medoid_cluster, 
    dh_rebalance_data
)
from datatrainer2 import (
    recursive_clustering, prepare_training_data, 
    prepare_validation_data, lstm_training
)
from modeleval import (
    dh_test_model, nam_dagostino_chi2, get_baseline_hazard_at_timepoints, combined_test_model
)

import psutil
torch.cuda.empty_cache()
gc.collect()

80

In [22]:
# Define Constants and Load Datasets
RANDOM_SEED = 12345
N_SPLIT = 2
FEATURE_COLS = ['gender', 'dm', 'ht', 'sprint', 'a1c', 'po4', 'UACR_mg_g', 'Cr', 'age', 'alb', 'ca', 'hb', 'hco3']
DURATION_COL = 'date_from_sub_60'
EVENT_COL = 'endpoint'
A_CLASS_COL = 'A_class'
G_CLASS_COL = 'G_class'
CLUSTER_COL = 'key'
TIME_GRID = np.array([i * 365 for i in range(6)])

# Define Feature Groups
CAT_FEATURES = ['gender', 'dm', 'ht', 'sprint']
LOG_FEATURES = ['a1c', 'po4', 'UACR_mg_g', 'Cr']
STANDARD_FEATURES = ['age', 'alb', 'ca', 'hb', 'hco3']
PASSTHROUGH_FEATURES = ['key', 'date_from_sub_60', 'endpoint']

# Load and Transform Data
BASE_FILENAME = '/mnt/d/pydatascience/g3_regress/data/X/X_20240628'
X_train_transformed, X_test_transformed = load_and_transform_data(
    BASE_FILENAME, CAT_FEATURES, LOG_FEATURES, STANDARD_FEATURES, PASSTHROUGH_FEATURES
)

2025-04-21 18:48:16,226 - INFO - Transforming training data...
2025-04-21 18:48:36,318 - INFO - Transforming test data...
/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [3]:
def create_neural_network(config, num_risk = len(X_train_transformed[EVENT_COL].unique()) - 1, num_time_bins=len(TIME_GRID)):
    """
    Function to create a neural network based on the given configuration.

    Args:
        config (dict): Configuration dictionary containing model type, network type, and hyperparameters.

    Returns:
        torch.nn.Module: Created neural network model.
    """
    gc.collect()
    torch.cuda.empty_cache()
    if config['model'] == 'deepsurv':
        num_risk = None
        num_time_bins=None
    elif config['model'] == 'deephit':
        num_risk = num_risk
        num_time_bins = num_time_bins
    # Create the Neural Network
    if config['net'] == 'ann':
        net = generalized_ann_net_init(
            input_size=len(config['features']),
            num_nodes=config["num_nodes"],
            batch_norm=config["batch_norm"],
            dropout=config["dropout"],
            output_size=1, # Default output size for DeepSurv
            num_risks = num_risk,
            num_time_bins = num_time_bins
        )
    elif config['net'] == 'lstm':
        net = lstm_net_init(
            input_size=len(config['features']),
            num_nodes=config["num_nodes"],
            batch_norm=config["batch_norm"],
            dropout=config["dropout"],
            num_risks = num_risk,
            num_time_bins = num_time_bins
        )
    else:
        raise ValueError("Unknown network type: {}".format(config['net']))

    optimizer = tt.optim.AdamWR(decoupled_weight_decay=1e-6, cycle_eta_multiplier=0.8)
    if config['model'] == 'deepsurv':
        model = CoxPH(net, optimizer)
    elif config['model'] == 'deephit':
        model = DeepHit(net, optimizer)
    model.optimizer.set_lr(config["lr"])
    
    return model

def train_neural_network(model, config, X_train, X_val, duration_col, event_col, cluster_col, callbacks, time_grid=None):
    """
    Function to train a given neural network using the provided datasets.

    Args:
        net (torch.nn.Module): Neural network to be trained.
        config (dict): Configuration dictionary containing model hyperparameters.
        X_train (pd.DataFrame): Training dataset with features.
        X_val (pd.DataFrame): Validation dataset with features.
        duration_col (str): Column representing event durations.
        event_col (str): Column representing event occurrences.
        cluster_col (str): Column for grouping during cross-validation.
        callbacks (list): List of callbacks for training.
        time_grid (np.array, optional): Time grid for evaluation if required. Defaults to None.

    Returns:
        model: Trained PyCox model.
        logs: Training logs.
    """
    gc.collect()
    torch.cuda.empty_cache()
    # Train the model
    if config['model'] == 'deepsurv':
        print('Initiate training of deepsurv neural network')
        X_val = df_event_focus(X_val, event_col, config['endpoint'])
        X_val_processed, y_val = preprocess_data(X_val, config['features'], duration_col, event_col)
        val_data = (X_val_processed, y_val)
        if config['net'] == 'ann':
            print('model structure: ANN')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                model, logs = recursive_clustering(model, X_train, duration_col, event_col, config, val_data, callbacks, max_repeats=30)
            elif config['balance_method'] == 'enn':
                print('data balancing method: smoteenn')
                X_train = rebalance_data(X_train, event_col, config['endpoint'], CAT_FEATURES, config, RANDOM_SEED, method='ENN')
                X_train, y_train = preprocess_data(X_train, config['features'], duration_col, event_col)
                logs = model.fit(X_train, y_train, config['batch_size'], int(config['max_epochs']), callbacks, verbose=True, val_data=val_data, num_workers=10)
            elif config['balance_method'] == 'tomek':
                print('data balancing method: smotetomek')
                X_train = rebalance_data(X_train, event_col, config['endpoint'], CAT_FEATURES, config, RANDOM_SEED, method='Tomek')
                X_train, y_train = preprocess_data(X_train, config['features'], duration_col, event_col)
                logs = model.fit(X_train, y_train, config['batch_size'], int(config['max_epochs']), callbacks, verbose=True, val_data=val_data, num_workers=10)
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)
            elif config['balance_method'] == 'NearMiss':
                print('data balancing method: NearMiss')
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)
    elif config['model'] == 'deephit':
        print('Initiate training of deephit neural network')
        X_val_processed, y_val = preprocess_data(X_val, config['features'], duration_col, event_col, TIME_GRID, discretize=True)
        val_data = (X_val_processed, y_val)
        if config['net'] == 'ann':
            print('model structure: ANN')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                model, logs = recursive_clustering(model, X_train, duration_col, event_col, config, val_data, callbacks, max_repeats=30, time_grid=TIME_GRID)
            elif config['balance_method'] == 'NearMiss':
                print('data balancing method: NearMiss')
                X_train = underbalance_data_general(X_train, EVENT_COL, CLUSTER_COL, config, version=config['version'])
                X_train, y_train = preprocess_data(X_train, config['features'], duration_col, event_col, TIME_GRID, discretize=True)
                logs = model.fit(X_train, y_train, config['batch_size'], int(config['max_epochs']), callbacks, verbose=True, val_data=val_data)
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)
            elif config['balance_method'] == 'NearMiss':
                print('data balancing method: NearMiss')
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)        

    # Free memory after training
    gc.collect()
    torch.cuda.empty_cache()

    return model, logs

def save_model(params, model, model_path, baseline_hazard_path):
    """
    Save model weights and baseline hazard data.

    Parameters:
    - model: The trained model to save.
    - model_path: Path to save the model weights (.pt file).
    - baseline_hazard_path: Path to save the baseline hazards (.pkl file).
    """
    # Compute baseline hazards and save
    if params['model'] == 'deepsurv':
        baseline_hazard = model.compute_baseline_hazards()
        baseline_hazard.to_pickle(baseline_hazard_path)
    
    # Save model weights
    model.save_model_weights(model_path)
    print(f"Model and baseline hazards saved to {model_path} and {baseline_hazard_path}.")

def training_wrapper(df, config, spliter, model_path, hazard_path, feature_col=FEATURE_COLS, duration_col=DURATION_COL, event_col=EVENT_COL, cluster_col=CLUSTER_COL, time_grid=TIME_GRID):
    """
    Train and save a survival analysis model with grouped cross-validation splits.

    This function performs training on grouped cross-validation splits of the input DataFrame and saves each trained model
    along with its baseline hazards. Memory management is handled to ensure efficient GPU usage.

    Parameters:
    - df (pd.DataFrame): DataFrame containing training data.
    - config (dict): Configuration dictionary for initializing the neural network.
    - spliter (object): Splitter object (e.g., GroupShuffleSplit or StratifiedKFold) used for creating train-validation splits.
    - model_path (str): File path to save the trained model weights (.pt file).
    - hazard_path (str): File path to save the baseline hazards (.pkl file).
    - feature_col (list): List of feature column names in `df` used for model training.
    - duration_col (str): Name of the column representing duration/time-to-event.
    - event_col (str): Name of the column representing the event indicator (0 = censored, 1 = event).
    - cluster_col (str): Name of the column used for grouping (clusters for cross-validation).
    - time_grid (list): List or array defining the time grid for training.

    Returns:
    - None: Saves the model weights and baseline hazard data for each cross-validation split.
    """
    for train_idx, val_idx in spliter.split(X=df[feature_col], y=df[event_col], groups=df[cluster_col]):
        # Clear GPU memory for each split
        gc.collect()
        torch.cuda.empty_cache()
        
        # Define early stopping callback
        callbacks = [tt.cb.EarlyStopping()]
        
        # Create training and validation sets
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        # Initialize and train the model
        model = create_neural_network(config)
        model, logs = train_neural_network(
            model, config,
            X_train=train_df, X_val=val_df,
            duration_col=duration_col, event_col=event_col,
            cluster_col=cluster_col, callbacks=callbacks, time_grid=time_grid
        )
        
        # Save the trained model and its baseline hazards
        save_model(config, model, model_path, hazard_path)
        
        # Free memory for the next iteration
        del model, logs
        gc.collect()
        torch.cuda.empty_cache()

    print("Training and saving completed for all cross-validation splits.")

    print("All models have been trained and saved successfully.")

In [12]:
model_ls = ['deepsurv_ann_clustering_1', 'deepsurv_ann_smoteenn_1', 'deepsurv_ann_smotetomek_1',
            'deepsurv_ann_clustering_2', 'deepsurv_ann_smoteenn_2', 'deepsurv_ann_smotetomek_2',
            'deepsurv_lstm_clustering_1', 'deepsurv_lstm_nearmiss_1', 'deepsurv_lstm_clustering_2', 'deepsurv_lstm_nearmiss_2',
            'deephit_ann_clustering_all', 'deephit_ann_nearmiss2_all', 'deephit_lstm_clustering_all', 'deephit_lstm_nearmiss1_all']
config_path = '/mnt/d/PYDataScience/g3_regress/code/models/all_model_configs.json'
model_path = '/mnt/d/PYDataScience/g3_regress/code/test/'
# Load the JSON file
with open(config_path, "r") as json_file:
    model_configs = json.load(json_file)

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
for train_idx_1, fin_val_idx in gss1.split(X=X_train_transformed[FEATURE_COLS], y=X_train_transformed[EVENT_COL], groups=X_train_transformed[CLUSTER_COL]):
    X_train_transformed_2, X_fin_val = X_train_transformed.iloc[train_idx_1, :], X_train_transformed.iloc[fin_val_idx, :]
    gc.collect()
    torch.cuda.empty_cache()
    for model in model_configs.keys():
        model_config = model_configs[model]
        if model_config is None:
            print(f"Configuration for {config_var_name} not found.")
            continue

        model_weights_path = f'{model_path}{model}.pt'
        model_hazard_path = f'{model_path}{model}_hazard.pkl'
        
        training_wrapper(X_train_transformed_2, model_config, gss2, model_weights_path, 
                        model_hazard_path, 
                        feature_col=FEATURE_COLS, duration_col=DURATION_COL, event_col=EVENT_COL, cluster_col=CLUSTER_COL, time_grid=TIME_GRID)
        gc.collect()
        torch.cuda.empty_cache()

2024-12-01 00:56:42,681 - INFO - Event column 'endpoint' updated with focus on event value 1.
2024-12-01 00:56:42,687 - INFO - Event column 'endpoint' updated with focus on event value 1.
2024-12-01 00:56:42,694 - INFO - Performing clustering iteration 1 / 20
2024-12-01 00:56:42,694 - INFO - CUDA environment set up and GPU memory cleared.
2024-12-01 00:56:42,698 - INFO - Event column 'endpoint' updated with focus on event value 1.


Initiate training of deepsurv neural network
model structure: ANN
data balancing method: clustering


2024-12-01 00:56:43,365 - INFO - Defined medoid for deepsurv model with 1207 clusters.


0:	[0s / 0s],		train_loss: 5.0903,	val_loss: 7.8046
1:	[0s / 0s],		train_loss: 5.0019,	val_loss: 7.4581
2:	[0s / 0s],		train_loss: 4.9083,	val_loss: 7.4617
3:	[0s / 0s],		train_loss: 4.8555,	val_loss: 7.1232
4:	[0s / 0s],		train_loss: 4.8227,	val_loss: 7.1393
5:	[0s / 0s],		train_loss: 4.8379,	val_loss: 7.0233
6:	[0s / 0s],		train_loss: 4.8391,	val_loss: 7.0490
7:	[0s / 0s],		train_loss: 4.8072,	val_loss: 7.1921
8:	[0s / 0s],		train_loss: 4.8053,	val_loss: 7.0329


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torchtuples/base.py:669: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.net.load_state_dict(torch.load(pat

...

2024-12-01 00:56:57,944 - INFO - Event column 'endpoint' updated with focus on event value 1.
2024-12-01 00:56:57,947 - INFO - Event column 'endpoint' updated with focus on event value 1.


Initiate training of deepsurv neural network
model structure: ANN
data balancing method: smoteenn


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/imblearn/over_sampling/_smote/base.py:370: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(
2024-12-01 00:57:02,181 - INFO - Missing values imputed using IterativeImputer.
2024-12-01 00:57:02,189 - INFO - Dataframe rebalanced with SMOTE and ENN.


0:	[10s / 10s],		train_loss: 3.6988,	val_loss: 5.0129
1:	[7s / 17s],		train_loss: 3.6731,	val_loss: 5.0671
2:	[7s / 24s],		train_loss: 3.6369,	val_loss: 5.0401
3:	[7s / 32s],		train_loss: 3.6515,	val_loss: 5.0541
4:	[7s / 39s],		train_loss: 3.6383,	val_loss: 5.0797
5:	[10s / 49s],		train_loss: 3.6292,	val_loss: 5.0409
6:	[7s / 57s],		train_loss: 3.6194,	val_loss: 5.0224
7:	[7s / 1m:4s],		train_loss: 3.6360,	val_loss: 5.0358


KeyboardInterrupt: 

In [13]:
def load_model(model, model_config, model_path, baseline_hazard_path):
    """
    Load model weights and baseline hazard data.

    Parameters:
    - create_model_func: Function to create the model architecture (e.g., create_neural_network).
    - model_path: Path to load the model weights (.pt file).
    - baseline_hazard_path: Path to load the baseline hazards (.pkl file).

    Returns:
    - model: The loaded model with weights and baseline hazards.
    """
    
    # Load model weights
    model.load_model_weights(model_path)
    
    # Load baseline hazards and assign to model
    if model_config['model'] == 'deepsurv':
        baseline_hazard = pd.read_pickle(baseline_hazard_path)
        model.baseline_hazards_ = baseline_hazard
        model.baseline_cumulative_hazards_ = baseline_hazard.cumsum()
    
    print(f"Model and baseline hazards loaded from {model_path} and {baseline_hazard_path}.")
    return model

In [14]:
def predict_neural_network(model, config, X_test, duration_col, event_col, cluster_col, time_grid=None):
    """
    Function to train a given neural network using the provided datasets.

    Args:
        net (torch.nn.Module): Neural network to be trained.
        config (dict): Configuration dictionary containing model hyperparameters.
        X_train (pd.DataFrame): Training dataset with features.
        X_val (pd.DataFrame): Validation dataset with features.
        duration_col (str): Column representing event durations.
        event_col (str): Column representing event occurrences.
        cluster_col (str): Column for grouping during cross-validation.
        callbacks (list): List of callbacks for training.
        time_grid (np.array, optional): Time grid for evaluation if required. Defaults to None.

    Returns:
        model: Trained PyCox model.
        logs: Training logs.
    """
    gc.collect()
    torch.cuda.empty_cache()
    # Train the model
    if config['model'] == 'deepsurv':
        print('Initiate testing of deepsurv neural network')
        X_test = df_event_focus(X_test, event_col, config['endpoint'])
        if config['net'] == 'ann':
            print('model structure: ANN')
            X_test_processed, y_test = preprocess_data(X_test, config['features'], duration_col, event_col)
            surv = model.predict_surv_df(X_test_processed, batch_size=512)
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            X_test_processed, y_test = prepare_validation_data(X_test, config['features'], duration_col, event_col, config, cluster_col, config['model'], time_grid)
            X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
            y_test_tensor = (torch.tensor(y_test[0], dtype=torch.float32), torch.tensor(y_test[1], dtype=torch.float32))
            surv = model.predict_surv_df(X_test_tensor, batch_size=512)
    elif config['model'] == 'deephit':
        print('Initiate testing of deephit neural network')
        if config['net'] == 'ann':
            print('model structure: ANN')
            X_test_processed, y_test = preprocess_data(X_test, config['features'], duration_col, event_col, time_grid, discretize=True)
            surv = model.predict_cif(X_test_processed, batch_size=512)
            print('prediction complete, please note that prediction of deephit models are CIF.')
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            X_test_processed, y_test = prepare_validation_data(X_test, config['features'], duration_col, event_col, config, cluster_col, config['model'], time_grid)
            surv = model.predict_cif(X_test_processed, batch_size=512)
            print('prediction complete, please note that prediction of deephit models are CIF.')

    # Free memory after training
    gc.collect()
    torch.cuda.empty_cache()

    return surv, y_test

def align_to_time_grid(surv, time_grid):
    """
    Align the survival DataFrame to the closest indices of the time grid.

    Parameters:
        surv (pd.DataFrame): Survival probabilities DataFrame.
        time_grid (np.array): Array of target time points to align.

    Returns:
        aligned_surv (pd.DataFrame): Aligned survival probabilities.
    """
    # Convert the DataFrame's index to a NumPy array for fast computation
    surv_times = np.array(surv.index)
    
    # Find the closest time in the survival DataFrame for each time in the grid
    closest_indices = [np.argmin(np.abs(surv_times - t)) for t in time_grid]
    
    # Extract the rows corresponding to the closest times
    aligned_surv = surv.iloc[closest_indices].copy()
    
    # Reindex the DataFrame to match the time grid
    aligned_surv.index = range(len(time_grid))  # Standardize indices to 0, 1, 2, ...
    return aligned_surv

In [23]:
cif_array_labels = [
    "deepsurv_ann_clustering",
    "deepsurv_ann_enn",
    "deepsurv_ann_tomek",
    "deepsurv_lstm_clustering",
    "deepsurv_lstm_NearMiss",
    "deephit_ann_clustering",
    "deephit_ann_NearMiss",
    "deephit_lstm_clustering",
    "deephit_lstm_NearMiss",
]

def prediction_wrapper(df, feature_col, duration_col, event_col, a_class_col, g_class_col, cluster_col, time_grid, config_path, model_path, cif_array_labels):
    
    
    """
    Perform and output predictions, ground truth durations, and ground truth events.

    Args:
        df (DataFrame): Test dataset with a 'key' column.
        feature_cols (list): List of feature columns.
        duration_col (str): Column name for duration.
        event_col (str): Column name for event types.
        cluster_col (str): Column name for clustering (e.g., patient identifier).
        time_grid (list): Time points for evaluation.
        config_path (str): path storing each model configs.
        model_path (str): path storing model weights and hazards.
        cif_array_labels (list): list of model labels to locate cif predictions in output.
    """
    
    gc.collect()
    torch.cuda.empty_cache()
    # Step 1: load the models needed
    with open(config_path, "r") as json_file:
        model_configs = json.load(json_file)
    loaded_models = {}

    for model_name in model_configs.keys():
        model_config = model_configs[model_name]
        if model_config is None:
            print(f"Configuration for {model_name} not found.")
            continue
        model_weights_path = f'{model_path}{model_name}.pt'
        model_hazard_path = f'{model_path}{model_name}_hazard.pkl'
        create_model_func = lambda: create_neural_network(
            config=model_config,
            num_risk=len(df[event_col].unique()) - 1,
            num_time_bins=len(time_grid)
        )
        model = create_model_func()
        loaded_models[model_name] = load_model(model, model_config, model_weights_path, model_hazard_path)
        
    # Step 2: Prepare ground truth from df 
    _, y = preprocess_data(df, feature_col, duration_col, event_col, a_class_col, g_class_col, time_grid, discretize=True)
    
    # Step 3: Make prediction
    predict_cif_dict = {model: np.zeros((len(df[event_col].unique())-1, len(time_grid), df.shape[0])) for model in cif_array_labels}
    for model_name in loaded_models.keys():
        model = loaded_models[model_name]
        model_config = model_configs[model_name]
        surv, _ = predict_neural_network(
                model, model_config,
                df,
                duration_col, event_col, cluster_col,
                time_grid
            )
        # Align survival probabilities (if DeepSurv)
        if model_config['model'] == 'deepsurv':
            surv = align_to_time_grid(surv, TIME_GRID).values  # 2D array
                
            # Structure key dynamically
            key = f"deepsurv_{model_config['net']}_{model_config['balance_method']}"
            assert surv.shape == predict_cif_dict[key][model_config['endpoint']-1].shape
            predict_cif_dict[key][model_config['endpoint']-1] = 1 - surv
            
        # Handle DeepHit predictions
        elif model_config['model'] == 'deephit':
            surv = np.array(surv)  # Convert to numpy array
                
            # Structure key dynamically
            key = f"deephit_{model_config['net']}_{model_config['balance_method']}"
            assert surv.shape == predict_cif_dict[key].shape
            predict_cif_dict[key] = surv
    for key in predict_cif_dict.keys():
        assert predict_cif_dict[key].shape[-1] == y[0].shape[0] == y[1].shape[0]
    return predict_cif_dict, y

In [31]:
import h5py
import os
from sklearn.utils import resample
import gc
import torch


def single_bootstrap_iteration(
    iteration_idx,
    df,
    feature_col,
    duration_col,
    event_col,
    a_class_col, 
    g_class_col,
    cluster_col,
    time_grid,
    config_path,
    model_path,
    cif_array_labels,
    output_dir
):
    """
    Perform a single bootstrap iteration and save the result to a separate HDF5 file.

    Args:
        iteration_idx (int): The index of the bootstrap iteration.
        df (DataFrame): The input dataframe.
        feature_col (list): Feature columns.
        duration_col (str): Duration column.
        event_col (str): Event column.
        cluster_col (str): Cluster column.
        time_grid (list): Time grid.
        config_path (str): Path to model configurations.
        model_path (str): Path to model weights.
        cif_array_labels (list): List of CIF array labels.
        output_dir (str): Directory to save HDF5 results.
    """
    print(f"Bootstrap Iteration {iteration_idx + 1}")

    # Resample keys with replacement
    unique_keys = df[cluster_col].unique()
    resampled_keys = resample(unique_keys, replace=True)

    # Filter the data by resampled keys
    resampled_data = df[df[cluster_col].isin(resampled_keys)]
    print(f"Total rows in resampled data for iteration {iteration_idx + 1}: {len(resampled_data)}")

    # Perform predictions
    predictions, truth = prediction_wrapper(
        resampled_data,
        feature_col,
        duration_col,
        event_col,
        a_class_col, 
        g_class_col,
        cluster_col,
        time_grid,
        config_path,
        model_path,
        cif_array_labels
    )
    # Define unique file name for this iteration
    output_file = os.path.join(output_dir, f"bootstrap_iteration_{iteration_idx + 1}.h5")

    # Save results to a separate HDF5 file
    with h5py.File(output_file, "w") as hdf:
        # Save predictions
        pred_group = hdf.create_group("predictions")
        for key, value in predictions.items():
            pred_group.create_dataset(key, data=value)
        
        # Save durations and events
        hdf.create_dataset("durations", data=truth[0])
        hdf.create_dataset("events", data=truth[1])
        hdf.create_dataset("a_class", data=np.char.encode(truth[2], encoding='utf-8'))
        hdf.create_dataset("g_class", data=np.char.encode(truth[2], encoding='utf-8'))

    print(f"Saved bootstrap iteration {iteration_idx + 1} to {output_file}.")


def bootstrap_predictions(
    df,
    feature_col,
    duration_col,
    event_col,
    a_class_col, 
    g_class_col,
    cluster_col,
    time_grid,
    config_path,
    model_path,
    cif_array_labels,
    n_bootstrap,
    output_dir
):
    """
    Perform bootstrap iterations sequentially and save each result to a separate HDF5 file.

    Args:
        df (DataFrame): Input dataframe containing test data.
        feature_col (list): List of feature columns.
        duration_col (str): Column name for duration.
        event_col (str): Column name for events.
        cluster_col (str): Column name for patient clustering.
        time_grid (list): Time grid for predictions.
        config_path (str): Path to model configurations in JSON format.
        model_path (str): Path to directory containing model weights and hazards.
        cif_array_labels (list): List of CIF array labels (model names).
        n_bootstrap (int): Number of bootstrap iterations.
        output_dir (str): Directory to save the HDF5 files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for i in range(n_bootstrap):
        try:
            single_bootstrap_iteration(
                i,
                df,
                feature_col,
                duration_col,
                event_col,
                a_class_col, 
                g_class_col,
                cluster_col,
                time_grid,
                config_path,
                model_path,
                cif_array_labels,
                output_dir
            )
        except Exception as e:
            print(f"Error in bootstrap iteration {i + 1}: {e}")

    print(f"Bootstrap completed. Results saved to {output_dir}.")
    gc.collect()
    torch.cuda.empty_cache()


In [32]:
config_path = '/mnt/d/PYDataScience/g3_regress/code/models/all_model_configs.json'
output_path = "/mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250421_test/"
bootstrap_predictions(
    df=X_test_transformed,
    feature_col=FEATURE_COLS,
    duration_col=DURATION_COL,
    event_col=EVENT_COL,
    a_class_col=A_CLASS_COL, 
    g_class_col=G_CLASS_COL,
    cluster_col=CLUSTER_COL,
    time_grid=TIME_GRID,
    config_path=config_path,
    model_path="/mnt/d/PYDataScience/g3_regress/code/test/",
    cif_array_labels=cif_array_labels,
    n_bootstrap=500,
    output_dir=output_path
)


Bootstrap Iteration 1
Total rows in resampled data for iteration 1: 25253
Saved bootstrap iteration 1 to /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250421_test/bootstrap_iteration_1.h5.
Bootstrap Iteration 2
Total rows in resampled data for iteration 2: 25943
Saved bootstrap iteration 2 to /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250421_test/bootstrap_iteration_2.h5.
Bootstrap Iteration 3
Total rows in resampled data for iteration 3: 25890
Saved bootstrap iteration 3 to /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250421_test/bootstrap_iteration_3.h5.
Bootstrap Iteration 4
Total rows in resampled data for iteration 4: 25623
Saved bootstrap iteration 4 to /mnt/d/PYDataScience/g3_regress/data/results/bootstrap_predictions/20250421_test/bootstrap_iteration_4.h5.
Bootstrap Iteration 5
Total rows in resampled data for iteration 5: 25430
Saved bootstrap iteration 5 to /mnt/d/PYDataScience/g3_regress/data/results/bo